# Day 5 | Lab 5.4: RAG Evaluation — RAGAS + Custom Retrieval Metrics

**Duration:** ~1.5 hours

**Scenario.** *Meridian Wealth Partners* — Investment Policy Documents from Module 4 — preserved from the source notebook (already-banking). We build a complete RAG-evaluation harness: RAGAS for end-to-end generation quality, plus custom retrieval-only metrics (Hit Rate@K, MRR, Precision@K, Recall@K, nDCG@K) that RAGAS does not cover.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Build a labelled evaluation dataset (question · ground-truth answer · ground-truth chunk IDs) for a RAG pipeline.
2. Run **RAGAS** to compute *faithfulness*, *answer relevancy*, *context precision*, and *context recall*.
3. Implement **Hit Rate@K, MRR, Precision@K, Recall@K, and nDCG@K** from scratch.
4. Run a **K-value sensitivity analysis** to pick the right K for production.
5. Compare **basic RAG vs advanced RAG vs contextual retrieval** on the same dataset, ranked by metric.
6. Recognise the failure modes of LLM-as-Judge metrics and when to combine them with deterministic metrics.

**Tools.** RAGAS · LangChain v1 · `langchain-openai` · `gpt-4.1-mini` (judge) · OpenAI embeddings · FAISS · NumPy.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


## 1. Install Dependencies


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' 'langchain-community' 'langchain-text-splitters' 'ragas>=0.2' faiss-cpu numpy datasets python-dotenv


## 2. API Key Configuration


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Why RAG Evaluation Is Hard (and Different from IR Evaluation)

Traditional information-retrieval (IR) evaluation has labelled relevance judgements — a query is tied to a known set of relevant docs in a corpus, and metrics like Precision@K, Recall@K, MRR, and nDCG@K score the ranking. You can run those over your retriever — and you should — but they ignore *generation*.

RAG quality is the **product** of two qualities: did retrieval bring the right context, *and* did the LLM faithfully answer from it? You need both:
- **Retrieval-only metrics** (Hit Rate@K, MRR, nDCG@K) — deterministic, cheap, repeatable. Use them for retriever ablations and CI guardrails.
- **End-to-end LLM-as-Judge metrics** (RAGAS) — capture faithfulness, answer relevancy, and context-utilisation. Catch problems retrieval-only metrics miss (e.g. retriever found the right chunk but the LLM hallucinated anyway).

**This lab builds both halves of the harness.**

## 4. The 4 Core RAGAS Metrics

| Metric | What it measures | Needs ground truth? | How |
|---|---|---|---|
| **Faithfulness** | Does every claim in the answer follow from the retrieved context? | No | LLM extracts claims, judges each against context |
| **Answer Relevancy** | Does the answer address the question? | No | LLM generates *N* questions from the answer; cosine-sim against original question |
| **Context Precision** | Are the *top-ranked* retrieved chunks the most relevant ones? | Yes (ground-truth answer) | LLM judges each chunk's relevance; weighted by rank |
| **Context Recall** | Did retrieval bring back *all* the chunks needed to answer? | Yes (ground-truth answer) | LLM splits ground-truth answer into claims; checks coverage |

**Faithfulness vs Answer Relevancy can disagree.** A perfectly-faithful answer that never addresses the question scores high faithfulness, low relevancy. An answer that addresses the question but with hallucinated claims scores high relevancy, low faithfulness. You need both.

## 5. Build the Document Corpus — Meridian Wealth Partners (preserved scenario)

Five Investment Policy Document excerpts from Meridian Wealth Partners — synthetic but in the same shape as Module 4. We use this corpus throughout.

In [ ]:
from langchain_core.documents import Document

MERIDIAN_DOCS: list[Document] = [
    Document(page_content=(
        '# Equity Investment Policy — Meridian Wealth Partners\n\n'
        'For the Aggressive Growth portfolio, equity allocation must remain between 70% and 90% '
        'of total assets under management. Single-stock concentration is capped at 5% of AUM. '
        'No more than 25% may be allocated to any single GICS sector. Small-cap exposure is '
        'capped at 20% of equity allocation.'
    ), metadata={'doc_id': 'IPS-EQ-001', 'topic': 'equity'}),
    Document(page_content=(
        '# Fixed Income Policy\n\n'
        'Investment-grade bonds (BBB- or higher) must comprise at least 80% of fixed income holdings. '
        'Average portfolio duration is constrained between 4 and 8 years. High-yield exposure is '
        'capped at 15%, with a maximum single-issuer concentration of 2%. Credit-default-swap usage '
        'is prohibited in client portfolios.'
    ), metadata={'doc_id': 'IPS-FI-001', 'topic': 'fixed_income'}),
    Document(page_content=(
        '# Rebalancing Policy\n\n'
        'Portfolios are reviewed monthly. A rebalancing event is triggered when any asset class '
        'drifts more than 5 percentage points from its target weight. Tax-loss harvesting is '
        'evaluated quarterly for taxable accounts. Wash-sale rules are enforced via a 31-day '
        'replacement-security blacklist.'
    ), metadata={'doc_id': 'IPS-OPS-001', 'topic': 'operations'}),
    Document(page_content=(
        '# Risk Management Framework\n\n'
        'Portfolio-level Value-at-Risk (95% confidence, 1-day horizon) must not exceed 2% of AUM. '
        'Stress tests model 2008-style equity drawdowns and 2020-style liquidity shocks quarterly. '
        'Maximum allowable portfolio leverage is 1.0 (no leverage permitted in standard accounts).'
    ), metadata={'doc_id': 'IPS-RISK-001', 'topic': 'risk'}),
    Document(page_content=(
        '# ESG Integration Policy\n\n'
        'For ESG-mandated accounts, MSCI ESG ratings of BBB or higher are required for all equity '
        'holdings. Tobacco, controversial-weapons, and thermal-coal producers are explicitly '
        'excluded. ESG screening is applied at portfolio construction and reconfirmed at every '
        'rebalance. Exceptions require written approval from the CIO.'
    ), metadata={'doc_id': 'IPS-ESG-001', 'topic': 'esg'}),
]

for d in MERIDIAN_DOCS:
    print(f'{d.metadata["doc_id"]:<14} ({d.metadata["topic"]:<13}) — {len(d.page_content)} chars')


## 6. Build a Baseline RAG Pipeline

We split each policy doc into ~300-char chunks (so retrieval errors are visible — long chunks hide them), embed with `text-embedding-3-small`, and store in FAISS.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(MERIDIAN_DOCS)
# Stable chunk_ids so we can label ground-truth retrievals.
for i, c in enumerate(chunks):
    c.metadata['chunk_id'] = f'C{i:03d}'

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = FAISS.from_documents(chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

print(f'Chunks: {len(chunks)}')
for c in chunks[:3]:
    print(f'  {c.metadata["chunk_id"]} | {c.metadata["doc_id"]} | {c.page_content[:70]!r}')


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system',
     'You are a compliance analyst at Meridian Wealth Partners. Answer the question using ONLY '
     'the provided context. If the answer is not in the context, say so explicitly. Cite the '
     'doc_id when relevant.'),
    ('user', 'Context:\n{context}\n\nQuestion: {question}'),
])

judge_llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

def format_docs(docs):
    return '\n\n'.join(f'[{d.metadata["chunk_id"]} · {d.metadata["doc_id"]}] {d.page_content}' for d in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | judge_llm
    | StrOutputParser()
)

test_q = 'What is the maximum single-stock concentration for the Aggressive Growth portfolio?'
print('Q:', test_q)
print('A:', rag_chain.invoke(test_q))


## 7. Build the Evaluation Dataset (12 questions with ground truth)

A RAG eval dataset has three columns:
1. **`question`** — what the user asked.
2. **`ground_truth_answer`** — what a domain expert would write.
3. **`ground_truth_chunk_ids`** — the chunk IDs that *should* appear in retrieval (used by Hit Rate, MRR, Precision@K, Recall@K, nDCG@K).

RAGAS metrics that need ground truth (context precision, context recall) use the answer; retrieval-only metrics use the chunk IDs.

In [ ]:
# Helper to find which chunk_ids contain a substring (deterministic ground-truth labelling).
def chunks_containing(*needles: str) -> list[str]:
    return [c.metadata['chunk_id'] for c in chunks
            if any(n.lower() in c.page_content.lower() for n in needles)]

EVAL_QUESTIONS = [
    {'question': 'What is the maximum single-stock concentration for the Aggressive Growth portfolio?',
     'ground_truth_answer': 'Single-stock concentration is capped at 5% of AUM.',
     'ground_truth_chunk_ids': chunks_containing('single-stock concentration')},
    {'question': 'What is the equity allocation range for the Aggressive Growth portfolio?',
     'ground_truth_answer': 'Equity allocation must remain between 70% and 90% of AUM.',
     'ground_truth_chunk_ids': chunks_containing('70% and 90%', 'equity allocation')},
    {'question': 'What credit-rating threshold defines investment-grade bonds in the policy?',
     'ground_truth_answer': 'Investment-grade bonds are rated BBB- or higher and must comprise at least 80% of fixed income holdings.',
     'ground_truth_chunk_ids': chunks_containing('BBB-', 'investment-grade')},
    {'question': 'What is the maximum high-yield exposure allowed?',
     'ground_truth_answer': 'High-yield exposure is capped at 15%.',
     'ground_truth_chunk_ids': chunks_containing('high-yield')},
    {'question': 'When is a rebalancing event triggered?',
     'ground_truth_answer': 'When any asset class drifts more than 5 percentage points from its target weight.',
     'ground_truth_chunk_ids': chunks_containing('5 percentage points', 'drifts')},
    {'question': 'What is the wash-sale blacklist period?',
     'ground_truth_answer': 'A 31-day replacement-security blacklist.',
     'ground_truth_chunk_ids': chunks_containing('31-day', 'wash-sale')},
    {'question': 'What is the portfolio-level VaR limit?',
     'ground_truth_answer': 'Portfolio-level VaR (95% confidence, 1-day horizon) must not exceed 2% of AUM.',
     'ground_truth_chunk_ids': chunks_containing('Value-at-Risk', '2% of AUM')},
    {'question': 'Is leverage permitted in standard client accounts?',
     'ground_truth_answer': 'No — maximum allowable portfolio leverage is 1.0 (no leverage permitted in standard accounts).',
     'ground_truth_chunk_ids': chunks_containing('leverage', '1.0')},
    {'question': 'What MSCI ESG rating is required for ESG-mandated accounts?',
     'ground_truth_answer': 'MSCI ESG ratings of BBB or higher are required for all equity holdings.',
     'ground_truth_chunk_ids': chunks_containing('MSCI ESG')},
    {'question': 'Which industries are explicitly excluded under the ESG policy?',
     'ground_truth_answer': 'Tobacco, controversial-weapons, and thermal-coal producers are excluded.',
     'ground_truth_chunk_ids': chunks_containing('tobacco', 'thermal-coal')},
    {'question': 'What is the average portfolio duration band for fixed income?',
     'ground_truth_answer': 'Average portfolio duration is constrained between 4 and 8 years.',
     'ground_truth_chunk_ids': chunks_containing('duration', '4 and 8')},
    {'question': 'Are credit default swaps permitted in client portfolios?',
     'ground_truth_answer': 'No — credit-default-swap usage is prohibited in client portfolios.',
     'ground_truth_chunk_ids': chunks_containing('credit-default-swap', 'prohibited')},
]

for q in EVAL_QUESTIONS:
    print(f'Q: {q["question"]}')
    print(f'   gt chunks: {q["ground_truth_chunk_ids"]}')
print(f'\n{len(EVAL_QUESTIONS)} eval questions built.')


## 8. Run the RAG Pipeline over the Evaluation Set

For each question we capture: the generated answer, the retrieved chunks (their `chunk_id`s and their text). RAGAS will consume these in the next step.

In [ ]:
import time

rag_runs: list[dict] = []
for ex in EVAL_QUESTIONS:
    q = ex['question']
    docs = retriever.invoke(q)
    answer = rag_chain.invoke(q)
    rag_runs.append({
        'question': q,
        'answer': answer,
        'retrieved_chunk_ids': [d.metadata['chunk_id'] for d in docs],
        'retrieved_contexts': [d.page_content for d in docs],
        'ground_truth_answer': ex['ground_truth_answer'],
        'ground_truth_chunk_ids': ex['ground_truth_chunk_ids'],
    })
    time.sleep(0.05)

for r in rag_runs[:3]:
    print(f'Q: {r["question"]}')
    print(f'  retrieved: {r["retrieved_chunk_ids"]}')
    print(f'  gt:        {r["ground_truth_chunk_ids"]}')
    print(f'  answer:    {r["answer"][:120]!r}')
    print()


## 9. RAGAS Evaluation — Faithfulness, Answer Relevancy, Context Precision, Context Recall

RAGAS expects a `Dataset` of dicts with keys `question`, `answer`, `contexts`, `ground_truth`. Under the hood every metric calls an LLM (we configure the cheaper `gpt-4.1-mini` as the judge).

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(judge_llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

ragas_dataset = Dataset.from_list([
    {
        'question': r['question'],
        'answer': r['answer'],
        'contexts': r['retrieved_contexts'],
        'ground_truth': r['ground_truth_answer'],
    }
    for r in rag_runs
])

ragas_result = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm,
    embeddings=ragas_emb,
)
print(ragas_result)


In [ ]:
ragas_df = ragas_result.to_pandas()
ragas_df[['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']]


## 10. Interpreting RAGAS Scores — When Faithfulness and Relevancy Disagree

Look for rows where one metric is high and another low — those are the diagnostic cases:
- *Faithfulness 1.0, Relevancy low* → answer is correct but evades the question (LLM hedged).
- *Relevancy 1.0, Faithfulness low* → answer addresses the question but invents claims (hallucination).
- *Context Precision low, Context Recall 1.0* → retrieved everything we needed plus a lot we didn't (K too high).
- *Context Precision 1.0, Context Recall low* → retrieved only the most-similar chunk, missed required complements (K too low or chunks too small).

These four metrics give you a 2×2 grid that maps directly to actions: tune retrieval (precision/recall), tune the prompt (faithfulness), or tune the synthesis (relevancy).

## 11. Custom Retrieval Metrics — RAGAS Doesn't Cover These

RAGAS's context precision/recall use the LLM as a judge. They're useful, but they're slow, costly, and stochastic. For retriever ablations and CI guardrails you want **deterministic IR metrics** computed against ground-truth chunk IDs.

| Metric | Definition | When you want it |
|---|---|---|
| **Hit Rate@K** | Fraction of queries where ≥1 ground-truth chunk appears in top-K. | First-pass sanity check — "are we retrieving anything useful?" |
| **MRR (Mean Reciprocal Rank)** | Mean over queries of `1 / rank-of-first-relevant`. | Single relevant doc per query — captures *how high* we rank it. |
| **Precision@K** | `(# relevant in top-K) / K`. | Penalises padding the top-K with irrelevant chunks. |
| **Recall@K** | `(# relevant in top-K) / (total relevant)`. | Critical when answers need *multiple* supporting chunks. |
| **nDCG@K** | Discounted Cumulative Gain normalised to [0, 1]. | Multi-relevance ranking — rewards putting *more relevant* chunks higher. |

All five are deterministic, ~milliseconds to compute, and reproducible across runs.

In [ ]:
import numpy as np

def hit_rate_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    top_k = retrieved[:k]
    return 1.0 if any(r in top_k for r in relevant) else 0.0

def reciprocal_rank(retrieved: list[str], relevant: list[str]) -> float:
    for i, doc_id in enumerate(retrieved, 1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0

def precision_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    top_k = retrieved[:k]
    if not top_k:
        return 0.0
    return sum(1 for d in top_k if d in relevant) / k

def recall_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    return sum(1 for d in top_k if d in relevant) / len(relevant)

def ndcg_at_k(retrieved: list[str], relevant: list[str], k: int) -> float:
    # Binary relevance — gain is 1 if doc is relevant, 0 otherwise.
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    dcg = sum((1.0 if d in relevant else 0.0) / np.log2(i + 1)
              for i, d in enumerate(top_k, 1))
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

# Quick sanity test on a hand-built case.
test_retrieved = ['C001', 'C007', 'C002', 'C019']
test_relevant  = ['C002', 'C007']
for fn in (hit_rate_at_k, precision_at_k, recall_at_k, ndcg_at_k):
    print(f'{fn.__name__:<18} K=4 → {fn(test_retrieved, test_relevant, 4):.4f}')
print(f'reciprocal_rank      → {reciprocal_rank(test_retrieved, test_relevant):.4f}')


## 12. Compute Retrieval Metrics Across the Eval Set

We retrieve top-10 (so the same retrieval can score multiple K values without re-retrieving), then compute every metric for every question.

In [ ]:
BIG_K = 10
K_FOR_METRICS = 4  # report headline metrics at K=4

wide_retriever = vectorstore.as_retriever(search_kwargs={'k': BIG_K})

rows = []
for ex in EVAL_QUESTIONS:
    docs = wide_retriever.invoke(ex['question'])
    retrieved_ids = [d.metadata['chunk_id'] for d in docs]
    rel = ex['ground_truth_chunk_ids']
    rows.append({
        'question': ex['question'][:55] + '…',
        f'hit@{K_FOR_METRICS}':  hit_rate_at_k(retrieved_ids, rel, K_FOR_METRICS),
        'mrr':                   reciprocal_rank(retrieved_ids, rel),
        f'p@{K_FOR_METRICS}':    precision_at_k(retrieved_ids, rel, K_FOR_METRICS),
        f'r@{K_FOR_METRICS}':    recall_at_k(retrieved_ids, rel, K_FOR_METRICS),
        f'ndcg@{K_FOR_METRICS}': ndcg_at_k(retrieved_ids, rel, K_FOR_METRICS),
    })

import pandas as pd
metrics_df = pd.DataFrame(rows)
metrics_df


In [ ]:
# Aggregate across the eval set.
agg = metrics_df.drop(columns=['question']).mean().to_dict()
print(f'=== Retriever scorecard (K = {K_FOR_METRICS}) ===')
for name, val in agg.items():
    print(f'  {name:<10} {val:.4f}')


## 13. K-Value Sensitivity Analysis

K is the most consequential retrieval hyperparameter. Too small → low recall; too big → low precision and a bigger context bill. Sweep K and chart the trade-off.

In [ ]:
import pandas as pd

K_VALUES = [1, 2, 3, 4, 5, 7, 10]

# Re-use top-10 retrievals; slice for each K.
retrieved_top10 = [
    [d.metadata['chunk_id'] for d in wide_retriever.invoke(ex['question'])]
    for ex in EVAL_QUESTIONS
]

sweep_rows = []
for k in K_VALUES:
    hits = np.mean([hit_rate_at_k(r, ex['ground_truth_chunk_ids'], k) for r, ex in zip(retrieved_top10, EVAL_QUESTIONS)])
    precs = np.mean([precision_at_k(r, ex['ground_truth_chunk_ids'], k) for r, ex in zip(retrieved_top10, EVAL_QUESTIONS)])
    recs = np.mean([recall_at_k(r, ex['ground_truth_chunk_ids'], k) for r, ex in zip(retrieved_top10, EVAL_QUESTIONS)])
    ndcgs = np.mean([ndcg_at_k(r, ex['ground_truth_chunk_ids'], k) for r, ex in zip(retrieved_top10, EVAL_QUESTIONS)])
    sweep_rows.append({'K': k, 'hit@K': hits, 'precision@K': precs, 'recall@K': recs, 'nDCG@K': ndcgs})

sweep_df = pd.DataFrame(sweep_rows).set_index('K')
sweep_df.round(4)


In [ ]:
import matplotlib.pyplot as plt

ax = sweep_df.plot(figsize=(7, 4), marker='o')
ax.set_title('K-value sensitivity — Meridian Wealth eval set')
ax.set_ylabel('metric value (averaged across queries)')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('\nReading the chart:')
print(' • hit@K and recall@K rise monotonically — more chunks, more chance of hitting a relevant one.')
print(' • precision@K typically falls — extra slots tend to be irrelevant.')
print(' • The elbow on nDCG@K is a reasonable production K — more chunks past the elbow give\n'
      '   diminishing ranking quality at the cost of context tokens.')


## 14. Comparison — Basic RAG vs Advanced Retrieval vs Contextual Retrieval

Labs 5.1 (advanced retrieval — multi-query, HyDE, hybrid, re-ranking) and 5.2 (contextual retrieval) build on the same Meridian corpus. The whole *point* of building this evaluation harness is to put those three approaches on the **same scoreboard**.

**Note** — Labs 5.1 and 5.2 may not be present in your worktree yet (they're being built by a parallel agent). Below we **simulate** their behaviour with two retriever variants we can build right here, so the benchmark cell runs end-to-end. Replace the simulated retrievers with the real Lab 5.1 / Lab 5.2 retrievers when those labs are available.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Variant A: basic (already built) — `retriever`, k=4.
basic_retriever = retriever

# Variant B: 'advanced' simulator — multi-query (proxy for Lab 5.1's strongest pattern).
advanced_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={'k': 4}),
    llm=judge_llm,
)

# Variant C: 'contextual' simulator — hybrid BM25 + dense (proxy for Lab 5.2's chunk-context augmentation
# wins, both push retrieval into a more recall-dominant regime).
bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 4
contextual_retriever = EnsembleRetriever(
    retrievers=[bm25, vectorstore.as_retriever(search_kwargs={'k': 4})],
    weights=[0.4, 0.6],
)

VARIANTS = {
    'basic_RAG (Lab 4.3 proxy)':       basic_retriever,
    'advanced_RAG (Lab 5.1 proxy)':    advanced_retriever,
    'contextual_RAG (Lab 5.2 proxy)':  contextual_retriever,
}


In [ ]:
# Score each variant on the deterministic retrieval-only metrics (cheap, reproducible).
K = 4
comparison = []
for name, ret in VARIANTS.items():
    scores = {'hit@4': [], 'mrr': [], 'p@4': [], 'r@4': [], 'ndcg@4': []}
    for ex in EVAL_QUESTIONS:
        docs = ret.invoke(ex['question'])
        rids = [d.metadata['chunk_id'] for d in docs]
        rel  = ex['ground_truth_chunk_ids']
        scores['hit@4'].append(hit_rate_at_k(rids, rel, K))
        scores['mrr'].append(reciprocal_rank(rids, rel))
        scores['p@4'].append(precision_at_k(rids, rel, K))
        scores['r@4'].append(recall_at_k(rids, rel, K))
        scores['ndcg@4'].append(ndcg_at_k(rids, rel, K))
    comparison.append({'variant': name, **{k: float(np.mean(v)) for k, v in scores.items()}})

comparison_df = pd.DataFrame(comparison).set_index('variant').round(4)
comparison_df['avg'] = comparison_df.mean(axis=1).round(4)
comparison_df = comparison_df.sort_values('avg', ascending=False)
comparison_df


**Reading the comparison table.** The variant with the highest *average* across hit@4, MRR, p@4, r@4, and nDCG@4 wins on retrieval. Plug each variant's retrieved contexts back into the RAGAS evaluator (Section 9) to score *generation* quality — sometimes a worse-on-retrieval variant wins on faithfulness because its answers are crisper. Always rank on the metric that maps to the production failure you're trying to fix.

## 15. When LLM-as-Judge Fails — Known Biases

RAGAS, custom LLM judges, and pairwise model evals all share the same failure modes. Memorise these:

| Bias | What happens | Mitigation |
|---|---|---|
| **Position bias** | Judge prefers the answer in position A (or B) regardless of quality. | Randomise position; run both orders and average. |
| **Verbosity bias** | Judge prefers longer answers, often inaccurately. | Score length-normalised; cap answer length. |
| **Self-enhancement bias** | Judge prefers answers that look like its own style. | Use a *different* model as judge than the one being evaluated. |
| **Sycophancy** | Judge agrees with the framing of the question. | Use neutral, structured rubrics; never frame the 'right' answer in the prompt. |
| **Family-resemblance bias** | Same model family scores higher than different family. | Cross-family judge ensembles (Claude judge for OpenAI outputs, etc). |
| **Prompt-leak bias** | Reasoning visible in the answer biases the judge upward. | Strip CoT before judging; judge final answers only. |

The deterministic retrieval metrics (Section 11) sidestep all of these — that's why a serious RAG eval harness combines *both*.

## 16. Reference — How OpenAI Evaluates RAG

OpenAI's internal evals decompose each answer into atomic claims, judge each claim against the context independently, and aggregate. This is the same shape as RAGAS's faithfulness metric — the public RAGAS implementation is a faithful re-implementation of that pattern. See the OpenAI Cookbook *RAG Evaluation* notebook for details.


---
## 17. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **RAGAS 4 metrics** | Faithfulness, answer relevancy, context precision, context recall — covers the answer-quality half of RAG. |
| **Retrieval-only metrics** | Hit@K, MRR, Precision@K, Recall@K, nDCG@K — deterministic, cheap, perfect for ablations and CI. |
| **Eval dataset** | (question, ground-truth answer, ground-truth chunk IDs) is the minimum schema. Build it deterministically with substring labelling on a small corpus; LLM-bootstrap on a large one. |
| **K sensitivity** | K is the single most consequential retrieval lever. Sweep it; pick the elbow on nDCG. |
| **Variant comparison** | Score basic / advanced / contextual on the *same* eval set with the *same* metrics — that's the only way to defend a recommendation. |
| **LLM-as-Judge biases** | Position, verbosity, self-enhancement, sycophancy, family-resemblance, prompt-leak. Combine with deterministic metrics. |

**Next Lab:** End of Module 5 — next module is Day 6: Claude API & AWS Bedrock.


## 18. Stretch Exercise (Optional)

1. Wire the *real* Lab 5.1 advanced retriever (HyDE + cross-encoder re-rank) into the comparison cell once Lab 5.1 is available, and re-rank the variants.
2. Add **answer correctness** as a custom metric: cosine similarity between embedded `answer` and embedded `ground_truth_answer`. Compare it to RAGAS faithfulness and discuss where they disagree.
3. Replace the deterministic substring labelling in Section 7 with an LLM-bootstrapped approach: feed each chunk to the LLM and ask 'list questions answerable from this chunk'. Measure the labelling overlap with the deterministic version.
4. Reduce K to 1 and re-run RAGAS — note which questions now fail context_recall, and trace whether the failure is at retrieval (chunk missing) or at synthesis (chunk found but ignored).
5. Add a *negative* eval question — 'What is the policy on cryptocurrency holdings?' — for which the corpus has no answer. Confirm RAGAS flags it (faithfulness should be high if the answer correctly says 'not in the context').


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. What are the 4 RAGAS metrics and what does each measure?**

*Hint:* Two on retrieval, two on generation.

*Answer sketch:* **Faithfulness** — every claim in the answer follows from the retrieved context (hallucination check). **Answer Relevancy** — the answer addresses the question. **Context Precision** — the top-ranked retrieved chunks are the most relevant ones. **Context Recall** — retrieval brought back all chunks needed to answer. The first two need no ground truth; the last two need a ground-truth answer.

---

**Q2. Hit Rate@K vs MRR vs nDCG — what does each capture, and when use which?**

*Hint:* Binary success vs first-rank vs graded ranking.

*Answer sketch:* **Hit@K** — binary: did *any* relevant doc appear in top-K? Use as a sanity floor. **MRR** — `1/rank-of-first-relevant`; rewards putting the *first* relevant doc near the top; use when one relevant doc per query suffices. **nDCG@K** — graded discounted gain normalised; rewards putting *more* relevant docs higher; use when relevance is multi-level or you need multiple supporting chunks.

---

**Q3. What is the difference between context precision and context recall?**

*Hint:* Precision punishes irrelevant chunks; recall punishes missing chunks.

*Answer sketch:* Context **precision** = of the top-K we retrieved, how many were actually relevant (rank-weighted). Context **recall** = of the relevant chunks that exist, how many did we retrieve. Low precision → K too high or retriever too noisy. Low recall → K too low, chunks too small, or retriever missing the right doc entirely.

---

**Q4. How do you build a ground-truth eval dataset for RAG without human labelers?**

*Hint:* Two passes: deterministic for small corpora, LLM-bootstrap for large.

*Answer sketch:* On a small/known corpus: substring-label the chunks (a chunk is 'relevant' to a question if it contains the answer phrase). On a large corpus: feed each chunk to an LLM with the prompt 'list 1–3 questions this chunk answers'; collect (chunk_id, question) pairs; deduplicate near-duplicate questions. Always spot-check 10% manually before trusting the dataset.

---

**Q5. When does an LLM-as-Judge fail — what are the known biases?**

*Hint:* Position, verbosity, self-enhancement, sycophancy, family-resemblance, prompt-leak.

*Answer sketch:* Judges prefer answers in a fixed position (randomise & average), longer answers (length-normalise), answers that resemble their own style (cross-family judges), answers that align with the question's framing (neutral rubric), answers that look like their own model family, and answers whose chain-of-thought leaks into the final text. Combine LLM judging with deterministic metrics so you can detect drift.

---

**Q6. Faithfulness vs answer relevancy — give a case where they disagree.**

*Hint:* Hedge answers vs hallucinated answers.

*Answer sketch:* *Faithful but not relevant:* 'The provided context discusses fixed-income duration ranges.' — every claim follows from the context, but the user asked about equity allocation. *Relevant but not faithful:* 'Maximum equity allocation is 95%' for a portfolio whose policy says 90%. Addresses the question but invents the number. RAGAS catches both because it scores them on different axes.

---

**Q7. How does K affect each retrieval metric — what's a good K for production?**

*Hint:* Hit and recall rise; precision falls; nDCG has a sweet spot.

*Answer sketch:* Hit@K and recall@K are monotonically non-decreasing in K — bigger K, never worse. Precision@K typically falls — extra slots are usually irrelevant. nDCG@K rises then plateaus or falls slightly as low-relevance docs accumulate. The production K is the elbow on the nDCG curve subject to context-budget constraints; for short factual questions K = 3–5 is typical, for multi-aspect synthesis K = 8–12.

---

**Q8. What's the difference between RAGAS evaluation and traditional IR evaluation?**

*Hint:* Generation-aware vs retrieval-only.

*Answer sketch:* Traditional IR (TREC, BEIR) scores ranked retrieval against relevance judgements — purely retrieval. RAGAS extends this with generation-aware metrics (faithfulness, answer relevancy) that need an LLM judge. RAGAS is a *superset*: you can and should still compute Hit@K / MRR / nDCG (deterministic IR) on the retriever, and RAGAS on the full pipeline. They're complementary; RAGAS alone can't tell you whether a regression is in retrieval or in synthesis.

